# 01. Data Exploration & Synthetic Data Generation

In this notebook, we explore the historical price data for Onion, Potato, and Tomato. Since real API limits may apply, we generate a robust synthetic dataset mimicking real Indian agriculture markets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import os

# Set random seed for reproducibility
np.random.seed(42)

# Create directory for data
os.makedirs("../backend/data/raw", exist_ok=True)

## Generating Synthetic Data
We generate 3 years of daily prices with seasonal patterns and festival spikes.

In [ ]:
def generate_synthetic_prices(commodity, base_price, volatility, start_date="2021-01-01", days=1095):
    dates = [pd.to_datetime(start_date) + timedelta(days=i) for i in range(days)]
    
    # Base trend
    prices = [base_price]
    
    for i in range(1, days):
        # Add random walk with some mean reversion to base price
        change = np.random.normal(0, volatility)
        
        # Seasonality: Monsoon (Jul-Sep) causes spikes, Winter (Dec-Feb) causes drops
        month = dates[i].month
        if month in [7, 8, 9]:
            seasonality = base_price * 0.005 # Upward pressure
        elif month in [12, 1, 2]:
            seasonality = -base_price * 0.003 # Downward pressure
        else:
            seasonality = 0
            
        # Festival spikes (Oct-Nov Diwali)
        if month in [10, 11] and 15 <= dates[i].day <= 25:
            seasonality = base_price * 0.02
            
        new_price = prices[-1] * 0.98 + base_price * 0.02 + change + seasonality
        prices.append(max(base_price * 0.4, new_price)) # Prevent going too low
        
    df = pd.DataFrame({"date": dates, "price": prices})
    df["commodity"] = commodity
    return df

df_onion = generate_synthetic_prices("onion", 1500, 50)
df_potato = generate_synthetic_prices("potato", 900, 30)
df_tomato = generate_synthetic_prices("tomato", 2000, 80)

df_all = pd.concat([df_onion, df_potato, df_tomato])
df_all.to_csv("../backend/data/raw/synthetic_prices.csv", index=False)
print("Synthetic data generated with shape:", df_all.shape)

## Visualizing the Data

In [ ]:
plt.figure(figsize=(15, 6))
for commodity in ["onion", "potato", "tomato"]:
    data = df_all[df_all["commodity"] == commodity]
    plt.plot(data["date"], data["price"], label=commodity.capitalize())
plt.title("Synthetic Agricultural Prices (3 Years)")
plt.xlabel("Date")
plt.ylabel("Price (₹/quintal)")
plt.legend()
plt.savefig("01_data_exploration.png")
plt.show()